<a href="https://colab.research.google.com/github/prroud/AI_learning_journey/blob/main/CatsAndDogsModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a model to classify cats and dogs from image. I will use transfer learning with *resnet18* model

## **1. Data**

In [88]:
!pip install kagglehub
!pip install torchmetrics

In [89]:
import kagglehub

path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dog-and-cat-classification-dataset' dataset.
Path to dataset files: /kaggle/input/dog-and-cat-classification-dataset


In [90]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision import models, transforms, datasets
import copy
from sklearn.model_selection import train_test_split
import os
from torchmetrics.classification import BinaryAccuracy


In [91]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
)

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
)

In [92]:
data_dir = f"{path}/PetImages"

full_train_dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)
full_val_dataset = datasets.ImageFolder(root=data_dir, transform=val_transform)

targets = full_train_dataset.targets
indices = list(range(len(targets)))

train_indices, test_indices = train_test_split(indices, test_size=0.2, stratify=targets, random_state=42)

train_dataset = Subset(dataset=full_train_dataset, indices=train_indices)
val_dataset = Subset(dataset=full_val_dataset, indices=test_indices)

train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True, num_workers=os.cpu_count())
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count())

In [93]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [94]:
print("Klasy wykryte przez ImageFolder:", full_train_dataset.classes)
print("Mapowanie klas:", full_train_dataset.class_to_idx)

Klasy wykryte przez ImageFolder: ['Cat', 'Dog']
Mapowanie klas: {'Cat': 0, 'Dog': 1}


## **2. Model**

In [95]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(in_features=num_features, out_features=1)
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params=model.fc.parameters(), lr=1e-3)

In [96]:
def train_model(model, criterion, optimizer, num_epochs=5):
    accuracy = BinaryAccuracy().to(device)

    dataloaders = {
        "train": train_dataloader,
        "val": val_dataloader
    }

    dataset_sizes = {
        "train": len(train_dataloader.dataset),
        "val": len(val_dataloader.dataset)
    }

    best_weights = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch: {epoch+1}/{num_epochs}")

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            accuracy.reset()

            for X_batch, y_batch in dataloaders[phase]:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device).float().unsqueeze(dim=1)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase=="train"):
                    y_logits = model(X_batch)
                    loss = criterion(y_logits, y_batch)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * X_batch.size(0)
                accuracy.update(torch.sigmoid(y_logits), y_batch)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = accuracy.compute().item()

            print(f"{phase} loss: {epoch_loss:.4f}, accuracy: {epoch_acc:.2f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = copy.deepcopy(model.state_dict())

        print('\n')

    print(f"Best val accuracy: {best_acc:.2f}")
    model.load_state_dict(best_weights)
    return model

In [97]:
model = train_model(model=model, criterion=criterion, optimizer=optimizer, num_epochs=5)

Epoch: 1/5


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train loss: 0.2205, accuracy: 0.91
val loss: 0.0622, accuracy: 0.98


Epoch: 2/5


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train loss: 0.1530, accuracy: 0.94
val loss: 0.0470, accuracy: 0.98


Epoch: 3/5


...

train loss: 0.1528, accuracy: 0.93
val loss: 0.0439, accuracy: 0.98


Epoch: 4/5


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train loss: 0.1443, accuracy: 0.94
val loss: 0.0514, accuracy: 0.98


Epoch: 5/5


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train loss: 0.1426, accuracy: 0.94
val loss: 0.0439, accuracy: 0.98


Best val accuracy: 0.98


In [ ]:
for name, child in model.named_children():
    if name in ["layer3", "layer4"]:
        for param in child.parameters():
            param.requires_grad = True

optimizer_fine = torch.optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4},
])

model = train_model(model=model, criterion=criterion, optimizer=optimizer_fine, num_epochs=5)

Epoch: 1/5


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train loss: 0.1276, accuracy: 0.95
val loss: 0.0303, accuracy: 0.99


Epoch: 2/5


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train loss: 0.1082, accuracy: 0.95
val loss: 0.0246, accuracy: 0.99


Epoch: 3/5


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train loss: 0.0957, accuracy: 0.96
val loss: 0.0246, accuracy: 0.99


Epoch: 4/5
